# Import and Setup

In [1]:
import os
import json
import ast
from openai import OpenAI

openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key
client = OpenAI()
MODEL = "gpt-5.4"

# LLM Functions

In [2]:
REQUIREMENTS = """- Must run quickly on a MacBook with 36GB RAM (Apple Silicon); use device='mps' where supported
- Use a single small transformer-based model (e.g. distilbert, all-MiniLM-L6-v2, or similarly lightweight models via transformers.pipeline or sentence-transformers)
- No training, fine-tuning, or weight updates — load a pretrained model and evaluate it directly (zero-shot or task-specific pretrained checkpoint)
- No hyperparameter tuning or loops over multiple models/configurations — pick one and run it
- Only one model and one dataset/subset
- Only code cells (no markdown cells)
- No plots or visualizations"""

In [3]:
def generate_data_science_tasks(n: int = 10) -> list:
    prompt = f"""Brainstorm a list of {n} descriptions of AI tasks that can be evaluated using a modern AI model and HuggingFace datasets.

Requirements for each task:
{REQUIREMENTS}
- Restrict to tasks with datasets that have less than a million samples
- Each description should specify both the task type and the dataset

Return ONLY a valid Python list of strings, no explanation."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content.strip()
    return ast.literal_eval(raw)


tasks = generate_data_science_tasks(10)
tasks

["Sentiment classification on the imdb dataset using a lightweight pretrained DistilBERT sentiment-analysis pipeline, evaluated directly on a small test subset with device='mps' when available.",
 'Emotion classification on the dair-ai/emotion dataset using a small pretrained text-classification transformer checkpoint, evaluated zero-shot-free as a direct label prediction task on the validation split.',
 'Natural language inference on the glue subset mnli using a compact pretrained DeBERTa or DistilBERT-style sequence-classification model, evaluated on the validation_matched split without any training.',
 'Paraphrase detection on the glue subset mrpc using a pretrained small sentence-pair classification transformer loaded from Hugging Face transformers, evaluated directly on the validation split.',
 'Linguistic acceptability classification on the glue subset cola using a lightweight pretrained transformer sequence-classification model, evaluated on the validation split on a MacBook wit

In [4]:
def generate_notebook(task: str, notebook_dir: str) -> str:
    # Step 0: Name the notebook
    name_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""Generate a short, descriptive filename for a Jupyter notebook about this AI task:

Task: {task}

Requirements:
- Use snake_case
- End with .ipynb
- Be concise (3-6 words)
- Return ONLY the filename, nothing else."""}],
    )
    notebook_name = name_response.choices[0].message.content.strip()
    if not notebook_name.endswith(".ipynb"):
        notebook_name += ".ipynb"
    print(f"=== [generate_notebook] Step 0: Name ===\n{notebook_name}\n")

    # Step 1: Plan
    plan_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Plan a Jupyter notebook workflow for this AI task:

Task: {task}

Requirements:
{REQUIREMENTS}

Write a concise step-by-step plan for the notebook that respects all requirements above."""}],
    )
    plan = plan_response.choices[0].message.content.strip()
    print(f"=== [generate_notebook] Step 1: Plan ===\n{plan}\n")

    # Step 2: Generate notebook JSON
    nb_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Generate a complete Jupyter notebook as valid JSON for this AI task.

Task: {task}

Plan:
{plan}

Requirements:
{REQUIREMENTS}
- Use HuggingFace datasets to load data
- Include cells for imports, data loading, inference, and evaluation
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}],
    )
    raw = nb_response.choices[0].message.content.strip()
    print(f"=== [generate_notebook] Step 2: Raw notebook JSON (first 500 chars) ===\n{raw[:500]}\n")

    if raw.startswith("```"):
        raw = raw.split("```", 2)[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.rsplit("```", 1)[0].strip()

    nb = json.loads(raw)
    os.makedirs(notebook_dir, exist_ok=True)
    path = os.path.join(notebook_dir, notebook_name)
    with open(path, "w") as f:
        json.dump(nb, f, indent=1)
    print(f"Notebook saved to {path}")
    return path

In [5]:
def generate_n_variations(notebook_path: str, n: int = 3, overwrite = False) -> list:
    with open(notebook_path, "r") as f:
        original_nb = json.load(f)

    cells_text = []
    for cell in original_nb.get("cells", []):
        source = "".join(cell.get("source", []))
        if source.strip():
            cells_text.append(source)
    original_content = "\n\n---\n\n".join(cells_text)

    original_name = os.path.splitext(os.path.basename(notebook_path))[0]
    notebook_dir = os.path.dirname(notebook_path)
    variations_dir = os.path.join(notebook_dir, f"variations_{original_name}")
    if os.path.exists(variations_dir) and not overwrite:
        return []
    os.makedirs(variations_dir, exist_ok=True)

    # Step 0: Plan all variations upfront, returning a name -> description dict
    plan_response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"""You are an expert data scientist. Plan {n} variations of the following Jupyter notebook.

Original notebook:
{original_content}

Requirements that every variation must satisfy:
{REQUIREMENTS}
- Stay on the same task type as the original notebook (e.g. if it is emotion classification, all variations must also be emotion classification)
- Do not switch to a different task type

Before writing the plans, think about the dimensions along which notebook implementations can meaningfully differ — such as which specific model checkpoint is used, how inference is performed, which portion or aspect of the data is evaluated, and what the outputs measure or report. Ensure the {n} variations are spread across these dimensions: no two variations should make the same choices on more than one such dimension.

Return ONLY a valid Python dictionary mapping a snake_case name to a concise description for each variation. Each description must explicitly state what makes it distinct from the original and from the other variations — covering the model, inference method, data handling, and evaluation outputs. No explanation outside the dict. The name will be the file name for the notebook without extensions."""}],
    )
    raw_plans = plan_response.choices[0].message.content.strip()
    print(f"=== [generate_n_variations] Step 0: Variation plans for '{original_name}' ===\n{raw_plans}\n")
    variation_plans = ast.literal_eval(raw_plans)

    paths = []
    for name, plan in variation_plans.items():
        print(f"--- [generate_n_variations] Generating variation: '{name}' ---")
        print(f"Plan: {plan}\n")

        # Step 1: Generate the variation notebook according to its plan
        nb_response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": f"""You are an expert data scientist. Generate a Jupyter notebook as valid JSON implementing this variation of an existing notebook.

Original notebook:
{original_content}

Variation plan:
{plan}

Requirements:
{REQUIREMENTS}
- Use HuggingFace datasets to load data
- Include cells for imports, data loading, inference, and evaluation
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}],
        )
        raw = nb_response.choices[0].message.content.strip()
        print(f"=== [generate_n_variations] Raw notebook JSON for '{name}' (first 300 chars) ===\n{raw[:300]}\n")

        if raw.startswith("```"):
            raw = raw.split("```", 2)[1]
            if raw.startswith("json"):
                raw = raw[4:]
            raw = raw.rsplit("```", 1)[0].strip()

        nb = json.loads(raw)
        variation_name = f"{name}.ipynb"
        path = os.path.join(variations_dir, variation_name)
        with open(path, "w") as f:
            json.dump(nb, f, indent=1)
        print(f"Variation saved to {path}")
        paths.append(path)

    return paths

# Initial Dataset

- Select best suited task from list
- Create initial notebook and store in "../notebooks/batch_2" folder.
- Create 5 variations of the initial notebook.
- For each variation, create 2-4 new variations of the variation.

In [17]:
selected_task = "Semantic textual similarity scoring on the STS-B validation set using the sentence-transformers/all-MiniLM-L6-v2 embedding model and cosine similarity correlation evaluation on device='mps'."

In [18]:
import shutil

NOTEBOOK_DIR = "../notebooks/batch_5"
if os.path.exists(NOTEBOOK_DIR):
    shutil.rmtree(NOTEBOOK_DIR)
os.makedirs(NOTEBOOK_DIR)

notebook_path = generate_notebook(selected_task, NOTEBOOK_DIR)

=== [generate_notebook] Step 0: Name ===
stsb_similarity_minilm_mps.ipynb

=== [generate_notebook] Step 1: Plan ===
```python
# 1) Install/import only the minimal libraries needed.
#    - Use sentence-transformers for all-MiniLM-L6-v2 embeddings
#    - Use datasets to load STS-B validation
#    - Use scipy / numpy / pandas for evaluation and compact result inspection
#    - Use torch to detect and use MPS if available
```

```python
# 2) Set up runtime configuration.
#    - Choose model_name = "sentence-transformers/all-MiniLM-L6-v2"
#    - Choose dataset = GLUE STS-B, validation split only
#    - Detect device with priority: "mps" if torch.backends.mps.is_available() else "cpu"
#    - Keep batch size modest for Apple Silicon, e.g. 64 or 128
#    - Set random seeds for reproducibility even though inference-only
```

```python
# 3) Load the STS-B validation split only.
#    - Load from datasets: load_dataset("glue", "stsb", split="validation")
#    - Keep only sentence1, sentence2, and 

In [19]:
variation_paths = generate_n_variations(notebook_path, n=7)

=== [generate_n_variations] Step 0: Variation plans for 'stsb_similarity_minilm_mps' ===
{
  "stsb_minilm_raw_cosine_baseline": "Uses the same lightweight sentence-transformers/all-MiniLM-L6-v2 checkpoint but changes inference to unnormalized embeddings with cosine similarity computed via sklearn/scipy-style vector norms instead of dot product on normalized vectors; keeps the full STS-B validation split; reports Pearson/Spearman for both raw cosine and linearly rescaled 0-5 predictions plus embedding dimensionality and runtime.",
  "stsb_mpnet_normalized_spearman_focus": "Switches to the small but stronger sentence-transformers/all-mpnet-base-v2 checkpoint; keeps dual-list sentence encoding with normalized embeddings and dot-product cosine on the full validation set; emphasizes rank-based evaluation by reporting Spearman, Pearson, score mean/std, and top/bottom prediction pairs, making it distinct in model choice and output analysis.",
  "stsb_distilroberta_pair_regression_pipeline": "

In [9]:


# for vp in variation_paths:
#     generate_n_variations(vp, n=7)

variation_paths = "../notebooks/batch_5/variations_stsb_similarity_minilm_mps"
for vp in os.listdir(variation_paths):
    if vp.endswith(".ipynb"):
        
        full_path = variation_paths + "/" + vp
        generate_n_variations(full_path, n=7)
        

=== [generate_n_variations] Step 0: Variation plans for 'stsb_minilm_length_bucket_subset' ===
{
  "stsb_minilm_full_validation_cosine_baseline": "Use sentence-transformers/all-MiniLM-L6-v2 on the full STS-B validation split without the original length-based filtering or 300-example cap; encode both sentence columns with SentenceTransformer.encode on MPS when available, compute normalized cosine similarity mapped to the 0-5 scale, and report Pearson, Spearman, MAE, RMSE, runtime, and the highest-error examples over the entire split.",
  "stsb_mpnet_filtered_quantile_slice": "Use sentence-transformers/all-mpnet-base-v2 as the single lightweight pretrained model, keep STS-B validation but replace the original deterministic shortest-first subset with a medium-length quantile slice after simple word-count filtering, run direct pairwise embedding inference via SentenceTransformer.encode with normalized embeddings, and report correlation metrics plus bucketed error summaries by sentence-leng

# Dataset Variation Test

In [20]:
selected_task = "Paraphrase detection on the GLUE MRPC dataset using a pretrained sentence-pair classification model such as DistilBERT, evaluated without any training."

In [25]:
import shutil

NOTEBOOK_DIR = "../notebooks/batch_4"
if os.path.exists(NOTEBOOK_DIR):
    shutil.rmtree(NOTEBOOK_DIR)
os.makedirs(NOTEBOOK_DIR)

notebook_path = generate_notebook(selected_task, NOTEBOOK_DIR)

=== [generate_notebook] Step 0: Name ===
mrpc_paraphrase_detection_distilbert.ipynb

=== [generate_notebook] Step 1: Plan ===
```python
# 1) Install/import only the minimal libraries needed.
#    Use:
#    - torch
#    - transformers
#    - datasets
#    - scikit-learn
#    Keep the notebook lightweight and fast.

# 2) Detect the best available device with Apple Silicon support.
#    Set:
#    - device = "mps" if torch.backends.mps.is_available()
#    - else "cpu"
#    Also print the chosen device for confirmation.

# 3) Choose exactly one small pretrained sentence-pair classification model.
#    Recommended single choice:
#    - "textattack/distilbert-base-uncased-MRPC"
#    Reason:
#    - small DistilBERT model
#    - already fine-tuned for MRPC-style paraphrase classification
#    - no training required
#    Load:
#    - AutoTokenizer
#    - AutoModelForSequenceClassification
#    Move model to the selected device and set model.eval().

# 4) Load exactly one dataset/subset from GLUE

In [6]:
variation_paths = generate_n_variations(notebook_path, n=7)

NameError: name 'notebook_path' is not defined

In [8]:
variation_paths = "../notebooks/batch_4/variations_mrpc_paraphrase_detection_distilbert"
for vp in os.listdir(variation_paths):
    if vp.endswith(".ipynb"):
        
        full_path = variation_paths + "/" + vp
        generate_n_variations(full_path, n=7)

=== [generate_n_variations] Step 0: Variation plans for 'mrpc_calibration_report' ===
{
  "mrpc_pipeline_subset_error_analysis": "Use the same task-specific lightweight checkpoint (textattack/distilbert-base-uncased-MRPC) but run inference through transformers.pipeline on MPS/CPU instead of manual batched forward passes; evaluate only the first 128 validation examples for faster execution; report accuracy, precision, recall, F1, confusion matrix, and a compact table of the top 10 highest-confidence mistakes plus top 10 lowest-confidence correct predictions, without calibration metrics.",
  "mrpc_manual_full_bert_tiny_calibration": "Switch to a different small paraphrase checkpoint, cross-encoder/nli-distilroberta-base on the GLUE MRPC validation set, using manual tokenization and batched model(**inputs) inference on MPS/CPU; keep the full validation split; report accuracy, precision, recall, F1, log loss, Brier score, expected calibration error, per-bin calibration summaries, and repre